# Customer Review Sentiment Analysis

Portfolio model: **TF-IDF + LinearSVC**. The notebook trains on your labeled CSV, saves the model, and opens a Streamlit app.

Run the cells in order. The app predicts one review from a text box. CSV upload is optional and is not required for that prediction.

`balanced_sentiment_dataset.csv` is the file used here. Its columns are `sentence` and `sentiment`. The dataset README defines `0` as Negative, `1` as Positive, and `2` as Neutral. Neutral rows in that file are synthetic examples.


## 1. Install dependencies

The next code cell installs the libraries. The cell after it writes `sentiment_core.py`, which the cleaning and training cells import. The Streamlit file is created later, after the model has been evaluated.


In [ ]:
%pip install -q "pandas>=2.0" "numpy>=1.24" "scikit-learn>=1.4" "matplotlib>=3.7" "joblib>=1.3" "streamlit>=1.32"


In [ ]:
%%writefile sentiment_core.py
"""Shared cleaning and LinearSVC training for customer-review sentiment.

TF-IDF is fitted only inside a scikit-learn Pipeline, and only on the
training split, so test reviews never influence the vocabulary.
"""

from __future__ import annotations

import json
import re
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

TEXT_PRIORITY = [
    "sentence",
    "review_text",
    "reviewtext",
    "review_body",
    "review",
    "customer_review",
    "tweet",
    "comment",
    "comments",
    "feedback",
    "text",
    "content",
    "body",
    "clean_text",
    "reviews",
]
LABEL_PRIORITY = [
    "sentiment",
    "sentiment_label",
    "polarity",
    "label",
    "emotion",
    "target",
    "class",
]
TEXT_LABELS = {
    "positive": "Positive",
    "pos": "Positive",
    "negative": "Negative",
    "neg": "Negative",
    "neutral": "Neutral",
    "neu": "Neutral",
}
# The supplied dataset README defines these codes.
CODE_LABELS_012 = {0: "Negative", 1: "Positive", 2: "Neutral"}
PIPELINE_NAME = "sentiment_pipeline.joblib"
VECTORIZER_NAME = "tfidf_vectorizer.joblib"
MODEL_NAME = "svm_model.joblib"
METRICS_NAME = "metrics.json"
ACCURACY_GOAL = 0.80


class ColumnDetectionError(Exception):
    """Raised when the review or sentiment column cannot be chosen safely."""


class UnclearLabelError(Exception):
    """Raised when label values do not match a known sentiment encoding."""


def read_dataset(path) -> pd.DataFrame:
    """Read a CSV, including files saved with a UTF-8 BOM or semicolon separator."""
    path = Path(path)
    if path.suffix.lower() not in {".csv", ".tsv", ".txt"}:
        raise ValueError("Please provide the dataset as a CSV file.")
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    last_error = None
    for encoding in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            frame = pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError as exc:
            last_error = exc
            continue
        if frame.shape[1] == 1:
            sample = path.read_text(encoding=encoding, errors="ignore")[:4000]
            if sample.count(";") > sample.count(","):
                frame = pd.read_csv(path, encoding=encoding, sep=";")
            elif sample.count("\t") > sample.count(","):
                frame = pd.read_csv(path, encoding=encoding, sep="\t")
        frame.columns = [str(column).strip() for column in frame.columns]
        if frame.columns.duplicated().any():
            duplicates = frame.columns[frame.columns.duplicated()].tolist()
            raise ValueError(f"The CSV has duplicate column names: {duplicates}")
        if frame.empty:
            raise ValueError("The CSV file has no rows.")
        return frame
    raise ValueError(f"Could not read {path.name} as text. Last error: {last_error}")


def detect_columns(frame: pd.DataFrame, text_col=None, label_col=None):
    """Return the review-text column and the sentiment column."""
    if text_col or label_col:
        if not text_col or not label_col:
            raise ColumnDetectionError(
                "Provide both the text column and the label column, or leave both empty for automatic detection."
            )
        missing = [name for name in (text_col, label_col) if name not in frame.columns]
        if missing:
            available = ", ".join(map(str, frame.columns))
            raise ColumnDetectionError(
                f"Column(s) not found: {', '.join(missing)}. Available columns: {available}"
            )
        if text_col == label_col:
            raise ColumnDetectionError("The text column and the label column must be different.")
        return text_col, label_col

    text_hits = _priority_hits(frame.columns, TEXT_PRIORITY)
    label_hits = _priority_hits(frame.columns, LABEL_PRIORITY)
    if len(text_hits) >= 1 and len(label_hits) >= 1 and text_hits[0] != label_hits[0]:
        return text_hits[0], label_hits[0]
    raise ColumnDetectionError(_column_help(frame))


def detect_text_column(frame: pd.DataFrame):
    """Return a likely review column, or None when the upload needs a manual choice."""
    hits = _priority_hits(frame.columns, TEXT_PRIORITY)
    return hits[0] if hits else None


def detect_label_column(frame: pd.DataFrame, text_col=None):
    """Return a likely sentiment column, or None when the upload has no label."""
    hits = [column for column in _priority_hits(frame.columns, LABEL_PRIORITY) if column != text_col]
    return hits[0] if hits else None


def clean_reviews(frame: pd.DataFrame, text_col: str, label_col: str):
    """Drop empty, duplicate, and contradictory rows and normalize sentiment labels."""
    report = {"rows_loaded": int(len(frame))}
    work = frame[[text_col, label_col]].copy()
    work.columns = ["text", "label_raw"]
    report["raw_label_counts"] = {
        _json_key(key): int(value)
        for key, value in work["label_raw"].value_counts(dropna=False).items()
    }
    work["text"] = work["text"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    raw_label_text = work["label_raw"].astype("string").str.strip()
    missing = work["text"].isna() | (work["text"] == "") | work["label_raw"].isna() | (raw_label_text == "")
    report["dropped_missing"] = int(missing.sum())
    work = work.loc[~missing].copy()
    if work.empty:
        raise ValueError("Every row was removed because the review text or the label was missing.")

    work["sentiment"] = normalize_labels(work["label_raw"])
    before_duplicates = len(work)
    work = work.drop_duplicates(subset=["text", "sentiment"])
    report["dropped_duplicates"] = int(before_duplicates - len(work))

    sentiment_per_text = work.groupby("text")["sentiment"].nunique()
    conflict_texts = sentiment_per_text[sentiment_per_text > 1].index
    conflict_rows = work["text"].isin(conflict_texts)
    report["dropped_conflicts"] = int(conflict_rows.sum())
    work = work.loc[~conflict_rows].copy()
    if work.empty:
        raise ValueError("Every row was removed because the same review had more than one sentiment label.")
    if work["sentiment"].nunique() < 2:
        raise ValueError("Cleaning left fewer than two sentiment classes, so there is nothing to train.")

    report["rows_clean"] = int(len(work))
    report["class_counts"] = {
        str(key): int(value) for key, value in work["sentiment"].value_counts().items()
    }
    return work[["text", "sentiment"]].reset_index(drop=True), report


def normalize_labels(raw: pd.Series) -> pd.Series:
    """Map known words and the dataset's 0/1/2 codes onto Positive, Negative, and Neutral."""
    text_mapped = raw.map(_map_text_label)
    if text_mapped.notna().all():
        return text_mapped

    numeric = pd.to_numeric(raw, errors="coerce")
    integer_like = numeric.notna() & (numeric == numeric.round())
    if integer_like.all():
        values = {int(value) for value in numeric.tolist()}
        table = _numeric_table(values)
        return numeric.map(lambda value: table[int(value)])

    unknown = raw.loc[text_mapped.isna()].astype(str).value_counts().head(12)
    examples = ", ".join(f"{label} ({count})" for label, count in unknown.items())
    raise UnclearLabelError(
        "Some label values are not a known sentiment encoding. "
        f"Examples: {examples}. "
        "Use Positive, Negative, and Neutral, or the dataset codes 0, 1, and 2."
    )


def train_and_evaluate(clean_frame: pd.DataFrame, test_size=0.2, random_state=42):
    """Fit TF-IDF and LinearSVC on the training split, then score the untouched test split."""
    label_counts = clean_frame["sentiment"].value_counts()
    if int(label_counts.min()) < 2:
        smallest = label_counts.idxmin()
        raise ValueError(
            f"The class '{smallest}' has only {int(label_counts.min())} row(s). "
            "Stratified splitting needs at least 2 rows in every class."
        )
    if not 0 < test_size < 0.5:
        raise ValueError("test_size must be between 0 and 0.5.")

    features = clean_frame["text"]
    labels = clean_frame["sentiment"]
    features_train, features_test, labels_train, labels_test = train_test_split(
        features,
        labels,
        test_size=test_size,
        random_state=random_state,
        stratify=labels,
    )
    pipeline = build_pipeline(len(features_train), random_state)
    started = time.perf_counter()
    try:
        pipeline.fit(features_train, labels_train)
    except ValueError as exc:
        raise ValueError(
            "The vectorizer could not build features from the training text. "
            "Check that the text column contains the reviews. "
            f"Original error: {exc}"
        ) from exc
    fit_seconds = time.perf_counter() - started

    test_predictions = pipeline.predict(features_test)
    train_predictions = pipeline.predict(features_train)
    class_names = [str(name) for name in pipeline.named_steps["svm"].classes_]
    precision, recall, f1, support = precision_recall_fscore_support(
        labels_test,
        test_predictions,
        labels=class_names,
        average=None,
        zero_division=0,
    )
    macro = precision_recall_fscore_support(
        labels_test,
        test_predictions,
        average="macro",
        zero_division=0,
    )
    weighted = precision_recall_fscore_support(
        labels_test,
        test_predictions,
        average="weighted",
        zero_division=0,
    )
    matrix = confusion_matrix(labels_test, test_predictions, labels=class_names)
    vectorizer = pipeline.named_steps["tfidf"]
    classifier = pipeline.named_steps["svm"]
    test_accuracy = float(accuracy_score(labels_test, test_predictions))

    per_class = {}
    for index, name in enumerate(class_names):
        per_class[name] = {
            "precision": float(precision[index]),
            "recall": float(recall[index]),
            "f1": float(f1[index]),
            "support": int(support[index]),
        }

    metrics = {
        "algorithm": "LinearSVC",
        "vectorizer_type": "TfidfVectorizer",
        "accuracy": test_accuracy,
        "train_accuracy": float(accuracy_score(labels_train, train_predictions)),
        "accuracy_goal": ACCURACY_GOAL,
        "meets_accuracy_goal": bool(test_accuracy >= ACCURACY_GOAL),
        "labels": class_names,
        "per_class": per_class,
        "macro_avg": _average_block(macro),
        "weighted_avg": _average_block(weighted),
        "confusion_matrix": matrix.astype(int).tolist(),
        "train_size": int(len(features_train)),
        "test_size": int(len(features_test)),
        "test_fraction": float(test_size),
        "random_state": int(random_state),
        "fit_seconds": round(fit_seconds, 2),
        "vectorizer": {
            "ngram_range": list(vectorizer.ngram_range),
            "min_df": vectorizer.min_df,
            "max_df": vectorizer.max_df,
            "max_features": vectorizer.max_features,
            "stop_words": "english, keeping negation words",
            "vocabulary_size": int(len(vectorizer.vocabulary_)),
        },
        "model": {
            "algorithm": "LinearSVC",
            "C": float(classifier.C),
            "class_weight": "balanced",
            "max_iter": int(classifier.max_iter),
        },
    }
    return pipeline, metrics


def finalize_metrics(metrics, source_path, text_col, label_col, clean_report):
    """Attach dataset facts to the evaluation numbers before they are saved."""
    source_path = Path(source_path)
    metrics["source_file"] = source_path.name
    metrics["sklearn_version"] = sklearn.__version__
    metrics["columns"] = {"text": text_col, "label": label_col}
    metrics["dataset"] = clean_report
    metrics["data_notes"] = data_notes(source_path)
    return metrics


def save_artifacts(pipeline, metrics, model_dir="models"):
    """Save the paired pipeline plus separate vectorizer and SVM files."""
    folder = Path(model_dir)
    folder.mkdir(parents=True, exist_ok=True)
    pipeline_path = folder / PIPELINE_NAME
    vectorizer_path = folder / VECTORIZER_NAME
    model_path = folder / MODEL_NAME
    metrics_path = folder / METRICS_NAME
    joblib.dump(pipeline, pipeline_path)
    joblib.dump(pipeline.named_steps["tfidf"], vectorizer_path)
    joblib.dump(pipeline.named_steps["svm"], model_path)
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    metrics["saved_files"] = [
        pipeline_path.as_posix(),
        vectorizer_path.as_posix(),
        model_path.as_posix(),
        metrics_path.as_posix(),
    ]
    return metrics


def load_artifacts(model_dir="models"):
    """Load the saved pipeline and the metrics written at training time."""
    folder = Path(model_dir)
    pipeline_path = folder / PIPELINE_NAME
    metrics_path = folder / METRICS_NAME
    if not pipeline_path.exists() or not metrics_path.exists():
        raise FileNotFoundError(
            "The trained model is not in the models folder yet. "
            "Run python train.py --data data/balanced_sentiment_dataset.csv"
        )
    pipeline = joblib.load(pipeline_path)
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    return pipeline, metrics


def run_training(data_path, text_col=None, label_col=None, model_dir="models", test_size=0.2, random_state=42):
    """Read, clean, train, evaluate, and save. This is the command-line entry path."""
    frame = read_dataset(data_path)
    text_name, label_name = detect_columns(frame, text_col, label_col)
    clean_frame, report = clean_reviews(frame, text_name, label_name)
    pipeline, metrics = train_and_evaluate(clean_frame, test_size=test_size, random_state=random_state)
    finalize_metrics(metrics, data_path, text_name, label_name, report)
    save_artifacts(pipeline, metrics, model_dir)
    return metrics


def format_report(metrics: dict) -> str:
    """Plain-text report of the real cleaning steps and held-out test metrics."""
    dataset = metrics["dataset"]
    lines = [
        f"Source file: {metrics['source_file']}",
        f"Text column: {metrics['columns']['text']}",
        f"Label column: {metrics['columns']['label']}",
        f"Rows loaded: {dataset['rows_loaded']}",
        f"Dropped missing review or label: {dataset['dropped_missing']}",
        f"Dropped duplicate rows: {dataset['dropped_duplicates']}",
        f"Dropped reviews with conflicting labels: {dataset['dropped_conflicts']}",
        f"Rows after cleaning: {dataset['rows_clean']}",
        "Raw label counts:",
    ]
    for key, value in dataset["raw_label_counts"].items():
        lines.append(f"  {key}: {value}")
    lines.append("Class counts after cleaning:")
    for label in metrics["labels"]:
        lines.append(f"  {label}: {dataset['class_counts'].get(label, 0)}")
    lines.extend(
        [
            "",
            (
                f"Split: stratified {1 - metrics['test_fraction']:.0%} train / "
                f"{metrics['test_fraction']:.0%} test, random_state={metrics['random_state']}"
            ),
            f"Train rows: {metrics['train_size']}",
            f"Test rows: {metrics['test_size']}",
            "Model: TF-IDF fitted on the training split only, then LinearSVC.",
            f"Vocabulary size: {metrics['vectorizer']['vocabulary_size']}",
            f"Fit time: {metrics['fit_seconds']} seconds",
            "",
            f"Test accuracy: {metrics['accuracy']:.4f}",
            f"Training accuracy, for comparison: {metrics['train_accuracy']:.4f}",
            f"Accuracy goal: {metrics['accuracy_goal']:.2f}",
        ]
    )
    if metrics["meets_accuracy_goal"]:
        lines.append("The held-out test accuracy meets the 80% goal.")
    else:
        lines.append(
            "The held-out test accuracy is below the 80% goal. The number above is the measured test result."
        )
    lines.extend(["", "Per-class test metrics:"])
    for label in metrics["labels"]:
        scores = metrics["per_class"][label]
        lines.append(
            f"  {label}: precision {scores['precision']:.4f}, "
            f"recall {scores['recall']:.4f}, f1 {scores['f1']:.4f}, support {scores['support']}"
        )
    lines.append(
        f"  Macro average: precision {metrics['macro_avg']['precision']:.4f}, "
        f"recall {metrics['macro_avg']['recall']:.4f}, f1 {metrics['macro_avg']['f1']:.4f}"
    )
    lines.extend(["", "Confusion matrix (rows are actual labels, columns are predicted labels):"])
    header = "  " + " ".join(f"{label:>12}" for label in metrics["labels"])
    lines.append(header)
    for label, row in zip(metrics["labels"], metrics["confusion_matrix"]):
        lines.append(f"  {label:>12} " + " ".join(f"{value:>12}" for value in row))
    if metrics.get("data_notes"):
        lines.extend(["", "Dataset notes:"])
        lines.extend(f"  - {note}" for note in metrics["data_notes"])
    lines.extend(["", "Saved files:"])
    lines.extend(f"  {path}" for path in metrics.get("saved_files", []))
    return "\n".join(lines)


def predict_reviews(pipeline, reviews) -> pd.DataFrame:
    """Predict sentiment for a list or series of reviews. Empty rows are skipped."""
    original = pd.Series(reviews, dtype="string").fillna("").astype(str)
    cleaned = original.str.replace(r"\s+", " ", regex=True).str.strip()
    classes = [str(name) for name in pipeline.named_steps["svm"].classes_]
    result = pd.DataFrame({"review": original.tolist()})
    result["predicted_sentiment"] = pd.NA
    result["decision_score"] = pd.NA
    for class_name in classes:
        result[f"score_{class_name}"] = pd.NA
    result["note"] = ""

    valid_positions = np.flatnonzero(cleaned.str.len().to_numpy() > 0)
    if len(valid_positions) == 0:
        result["note"] = "Empty review skipped"
        return result

    texts = cleaned.iloc[valid_positions].tolist()
    predicted = pipeline.predict(texts)
    matrix = _score_matrix(pipeline, texts)
    class_index = {name: index for index, name in enumerate(classes)}
    for row_number, position in enumerate(valid_positions):
        position = int(position)
        label = str(predicted[row_number])
        result.at[position, "predicted_sentiment"] = label
        result.at[position, "decision_score"] = round(float(matrix[row_number, class_index[label]]), 4)
        for class_name in classes:
            result.at[position, f"score_{class_name}"] = round(
                float(matrix[row_number, class_index[class_name]]),
                4,
            )
    empty_positions = np.flatnonzero(cleaned.str.len().to_numpy() == 0)
    for position in empty_positions:
        result.at[int(position), "note"] = "Empty review skipped"
    return result


def data_notes(source_path: Path) -> list[str]:
    """Notes taken from the README shipped with this dataset."""
    name = Path(source_path).name.lower()
    code_note = "Label codes from the dataset README: 0 = Negative, 1 = Positive, 2 = Neutral."
    if name == "balanced_sentiment_dataset.csv":
        return [
            "The saved model uses balanced_sentiment_dataset.csv. The dataset README recommends this subset for SVM experiments.",
            code_note,
            "The dataset README states that neutral messages are synthetic, while positive and negative rows are sampled from the original training data.",
            "expanded_train_data.csv was not used to fit the saved model.",
        ]
    if name == "expanded_train_data.csv":
        return [
            "The saved model uses expanded_train_data.csv. The dataset README says this file keeps the original rows and adds generated neutral messages.",
            code_note,
        ]
    return [code_note]


# These stay in the text. The English stop list would otherwise delete "not"
# and turn "not bad" into "bad".
_NEGATION_WORDS = frozenset(
    {
        "not",
        "no",
        "nor",
        "never",
        "neither",
        "none",
        "nobody",
        "nothing",
        "nowhere",
        "cannot",
        "without",
    }
)
_NEGATION_PATTERN = re.compile(
    r"\b(?:not|no|never|cannot|isn.?t|wasn.?t|weren.?t|aren.?t|don.?t|doesn.?t|didn.?t|"
    r"won.?t|can.?t|couldn.?t|wouldn.?t|shouldn.?t)\s+([a-z]+)",
    re.IGNORECASE,
)


def mark_negation(text: str) -> str:
    """Join a negation to the next word, so 'not bad' is one token instead of 'bad'."""
    lowered = str(text).lower()
    return _NEGATION_PATTERN.sub(lambda match: "not_" + match.group(1), lowered)


def sentiment_stop_words() -> list[str]:
    """English stop words with negation words left in the review text."""
    return sorted(ENGLISH_STOP_WORDS - _NEGATION_WORDS)


def build_pipeline(train_rows: int, random_state: int) -> Pipeline:
    """Create an unfitted TF-IDF + LinearSVC pipeline sized to the training split."""
    min_df = 2 if train_rows >= 200 else 1
    max_df = 0.95 if train_rows >= 200 else 1.0
    # The vocabulary cap is 30,000 features. Larger training sets use the primal SVM form.
    dual = train_rows < 30000
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    preprocessor=mark_negation,
                    stop_words=sentiment_stop_words(),
                    token_pattern=r"(?u)\b\w\w+\b",
                    ngram_range=(1, 2),
                    min_df=min_df,
                    max_df=max_df,
                    max_features=30000,
                    sublinear_tf=True,
                    dtype=np.float32,
                ),
            ),
            (
                "svm",
                LinearSVC(
                    C=1.0,
                    class_weight="balanced",
                    random_state=random_state,
                    max_iter=10000,
                    dual=dual,
                ),
            ),
        ]
    )


def _priority_hits(columns, priority):
    lookup = {_normalize_name(column): column for column in columns}
    return [lookup[name] for name in priority if name in lookup]


def _normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def _map_text_label(value):
    if pd.isna(value):
        return pd.NA
    text = re.sub(r"[^a-z]+", " ", str(value).strip().lower())
    text = re.sub(r"\s+", " ", text).strip()
    return TEXT_LABELS.get(text, pd.NA)


def _numeric_table(values: set[int]) -> dict[int, str]:
    if values <= {0, 1}:
        return {0: "Negative", 1: "Positive"}
    if values <= {0, 1, 2}:
        return CODE_LABELS_012
    if values <= {-1, 0, 1}:
        return {-1: "Negative", 0: "Neutral", 1: "Positive"}
    shown = ", ".join(str(value) for value in sorted(values))
    raise UnclearLabelError(
        f"The label column contains the numbers {shown}. "
        "Star ratings are left unchanged until you say how they should map. "
        "This dataset's README uses 0 = Negative, 1 = Positive, and 2 = Neutral."
    )


def _score_matrix(pipeline, texts):
    raw_scores = pipeline.decision_function(texts)
    if raw_scores.ndim == 1:
        column = np.asarray(raw_scores, dtype=float).reshape(-1, 1)
        return np.column_stack([-column, column])
    return np.asarray(raw_scores, dtype=float)


def _average_block(scores) -> dict:
    precision, recall, f1, _support = scores
    return {"precision": float(precision), "recall": float(recall), "f1": float(f1)}


def _column_help(frame: pd.DataFrame) -> str:
    lines = [
        "I could not tell which column is the review text and which column is the sentiment label.",
        "Columns in the file:",
    ]
    for column in frame.columns:
        sample = frame[column].dropna().astype(str).head(3).tolist()
        preview = "; ".join(value[:80] for value in sample) if sample else "(empty)"
        lines.append(f"- {column} ({frame[column].dtype}): {preview}")
    lines.append(
        "Run the trainer again with both names, for example: "
        "python train.py --data your.csv --text-col sentence --label-col sentiment"
    )
    return "\n".join(lines)


def _json_key(value) -> str:
    if pd.isna(value):
        return "missing"
    return str(value)


## 2. Upload or load the dataset

If `data/balanced_sentiment_dataset.csv` is already in the working directory, this cell uses it. Otherwise it asks you to upload a CSV. In Colab, choose `balanced_sentiment_dataset.csv` from the supplied zip.

Leave `TEXT_COLUMN` and `LABEL_COLUMN` empty to detect `sentence` and `sentiment`. Fill both only when detection stops and asks you to.


In [ ]:
from pathlib import Path

DATA_PATH = "data/balanced_sentiment_dataset.csv"
TEXT_COLUMN = ""  # set both names only if detection cannot tell the columns apart
LABEL_COLUMN = ""

Path("data").mkdir(exist_ok=True)
if Path(DATA_PATH).exists():
    print("Using", DATA_PATH)
else:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit(
            "Put the CSV in the data folder, or open this notebook in Google Colab to upload it."
        )
    print("Upload balanced_sentiment_dataset.csv from the dataset zip.")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file was uploaded.")
    name = next(iter(uploaded))
    destination = Path("data") / Path(name).name
    destination.write_bytes(uploaded[name])
    DATA_PATH = str(destination)
    print("Saved", DATA_PATH)


## 3. Inspect and preprocess

This cell prints the columns, missing values, and label counts, then removes incomplete rows, duplicate rows, and reviews that carry more than one sentiment.


In [ ]:
from sentiment_core import clean_reviews, detect_columns, read_dataset

frame = read_dataset(DATA_PATH)
print("Shape:", frame.shape)
print("Columns:", list(frame.columns))
display(frame.head())
print("Missing values:")
display(frame.isna().sum().to_frame("missing"))

text_col, label_col = detect_columns(frame, TEXT_COLUMN or None, LABEL_COLUMN or None)
print("Text column:", text_col)
print("Label column:", label_col)
print("Raw label counts:")
display(frame[label_col].value_counts(dropna=False).to_frame("rows"))

clean_df, clean_report = clean_reviews(frame, text_col, label_col)
print("Cleaning report:")
for key, value in clean_report.items():
    print(f"  {key}: {value}")
display(clean_df.head())


## 4. Train and evaluate the SVM

TF-IDF and `LinearSVC` sit in one pipeline. The vectorizer is fitted on the training split only. The test split is stratified and is not used during fitting. The printed accuracy is the measured test result.


In [ ]:
import pandas as pd
from sentiment_core import finalize_metrics, train_and_evaluate

pipeline, metrics = train_and_evaluate(clean_df)
finalize_metrics(metrics, DATA_PATH, text_col, label_col, clean_report)

print(f"Test accuracy: {metrics['accuracy']:.4f}")
print(f"Training accuracy: {metrics['train_accuracy']:.4f}")
print(f"Meets 80% goal: {metrics['meets_accuracy_goal']}")
display(pd.DataFrame(metrics["per_class"]).T)
labels = metrics["labels"]
display(
    pd.DataFrame(
        metrics["confusion_matrix"],
        index=[f"Actual {label}" for label in labels],
        columns=[f"Predicted {label}" for label in labels],
    )
)


## 5. Save the model and vectorizer

The app loads `models/sentiment_pipeline.joblib`, which keeps the fitted TF-IDF vectorizer and the LinearSVC model together. The vectorizer and SVM are also saved as separate files.


In [ ]:
from sentiment_core import format_report, save_artifacts

save_artifacts(pipeline, metrics, "models")
print(format_report(metrics))


## 6. Create the Streamlit application

This cell writes `app.py`. The page loads the saved pipeline and does not train again.


In [ ]:
%%writefile app.py
"""Streamlit dashboard for customer-review sentiment.

The page loads the saved TF-IDF + LinearSVC pipeline. It does not retrain.
"""

from __future__ import annotations

import html
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import sklearn
import streamlit as st

BASE_DIR = Path(__file__).resolve().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from sentiment_core import detect_text_column, load_artifacts, predict_reviews

STYLES = {
    "Positive": {"color": "#0B8A4B", "bg": "#E7F6EE"},
    "Negative": {"color": "#C4372F", "bg": "#FDECEC"},
    "Neutral": {"color": "#9A6B12", "bg": "#FFF6DE"},
}
UPLOAD_COLORS = {
    "Positive": "#0B8A4B",
    "Negative": "#C4372F",
    "Neutral": "#2F6FDB",
}
CARD_ORDER = ["Positive", "Negative", "Neutral"]
EXAMPLES = [
    "I love these headphones. The sound is excellent and they arrived quickly.",
    "The package arrived on Tuesday.",
    "This broke on the second day and support never replied.",
]


def main():
    site = website_name()
    st.set_page_config(
        page_title=site or "Customer Review Sentiment",
        page_icon="◆",
        layout="wide",
        initial_sidebar_state="expanded",
    )
    inject_css()
    try:
        pipeline, metrics = get_artifacts()
    except FileNotFoundError as exc:
        st.title("Customer Review Sentiment")
        st.error(str(exc))
        st.markdown(
            "Train the model once, then refresh this page. "
            "A single review prediction does not need a CSV upload."
        )
        st.code(
            "python train.py --data data/balanced_sentiment_dataset.csv",
            language="text",
        )
        return

    render_sidebar(metrics)
    render_header(metrics)
    render_cards(metrics)
    prediction_column, chart_column = st.columns([1.15, 0.85], gap="large")
    with prediction_column:
        render_single_prediction(pipeline)
    with chart_column:
        st.pyplot(distribution_figure(metrics), clear_figure=True)
        st.caption("Counts from the cleaned dataset saved during training.")
        st.pyplot(confusion_figure(metrics), clear_figure=True)
        st.caption("Rows are the actual test labels. Columns are the predicted labels.")
    render_bulk(pipeline)
    render_footer(metrics)


@st.cache_resource
def get_artifacts():
    return load_artifacts(BASE_DIR / "models")


def render_sidebar(metrics):
    with st.sidebar:
        st.header("Model")
        st.write("TF-IDF + LinearSVC")
        st.write(f"Source file: `{metrics.get('source_file', 'unknown')}`")
        st.write(f"Test accuracy: **{metrics['accuracy'] * 100:.2f}%**")
        st.write(f"Train rows: {metrics['train_size']:,}")
        st.write(f"Test rows: {metrics['test_size']:,}")
        st.write(f"Vocabulary: {metrics['vectorizer']['vocabulary_size']:,} features")
        saved_version = metrics.get("sklearn_version")
        if saved_version and saved_version != sklearn.__version__:
            st.warning(
                f"This model was saved with scikit-learn {saved_version}. "
                f"The app is running {sklearn.__version__}."
            )
        st.divider()
        st.header("Dataset notes")
        for note in metrics.get("data_notes", []):
            st.write(note)


def website_name() -> str:
    """Return the website name saved from the text field, if the user entered one."""
    return str(st.session_state.get("website_name", "")).strip()


def render_header(metrics):
    site = website_name()
    safe_site = html.escape(site)
    title = safe_site or "Customer Review Sentiment"
    detail = (
        f"Sentiment for reviews on {safe_site}. "
        "Type one review and the saved SVM model returns Positive, Negative, or Neutral."
        if site
        else "Type one review and the saved SVM model returns Positive, Negative, or Neutral."
    )
    st.markdown(
        f"""
        <div class="hero">
          <h1>{title}</h1>
          <p>{detail}</p>
        </div>
        """,
        unsafe_allow_html=True,
    )
    st.text_input(
        "Name of your website",
        key="website_name",
        placeholder="Enter the name of your website",
    )
    goal = "meets" if metrics["meets_accuracy_goal"] else "is below"
    st.caption(
        f"Held-out test accuracy is {metrics['accuracy'] * 100:.2f}% "
        f"({metrics['accuracy']:.4f}), which {goal} the 80% goal. "
        "The model is loaded from disk and is not retrained when this page opens."
    )


def render_cards(metrics):
    counts = metrics["dataset"]["class_counts"]
    total = int(metrics["dataset"]["rows_clean"])
    visible = [label for label in CARD_ORDER if label in counts]
    extras = [label for label in counts if label not in CARD_ORDER]
    labels = ["Total"] + visible + extras
    values = [total] + [int(counts[label]) for label in visible + extras]
    columns = st.columns(len(labels))
    for column, label, value in zip(columns, labels, values):
        color = STYLES.get(label, {}).get("color", "#1B2421")
        column.markdown(
            f"""
            <div class="stat-card">
              <p class="stat-label">{html.escape(label)}</p>
              <p class="stat-value" style="color:{color}">{value:,}</p>
            </div>
            """,
            unsafe_allow_html=True,
        )


def render_single_prediction(pipeline):
    st.subheader("Predict one review")
    st.write("CSV upload is optional and lives further down the page.")
    if "review_text" not in st.session_state:
        st.session_state.review_text = ""
    example_columns = st.columns(3)
    example_names = ["Positive example", "Neutral example", "Negative example"]
    for column, name, example in zip(example_columns, example_names, EXAMPLES):
        if column.button(name, use_container_width=True):
            st.session_state.review_text = example
            st.session_state.pop("last_result", None)
            st.rerun()

    with st.form("single-review"):
        review = st.text_area(
            "Customer review",
            key="review_text",
            height=150,
            placeholder="Paste a customer review here.",
        )
        submitted = st.form_submit_button("Predict sentiment", type="primary", use_container_width=True)

    if submitted:
        message = validate_review(review)
        if message:
            st.error(message)
        else:
            prediction = predict_reviews(pipeline, [review]).iloc[0]
            st.session_state.last_result = prediction.to_dict()

    result = st.session_state.get("last_result")
    if result:
        render_result(result)


def render_result(result):
    label = str(result["predicted_sentiment"])
    style = STYLES.get(label, {"color": "#334155", "bg": "#F1F5F9"})
    score = result["decision_score"]
    st.markdown(
        f"""
        <div class="result-banner" style="background:{style['bg']}; border-color:{style['color']}">
          <p class="result-kicker">Predicted sentiment</p>
          <p class="result-label" style="color:{style['color']}">{html.escape(label)}</p>
          <p class="result-score">Decision score {html.escape(str(score))}</p>
        </div>
        """,
        unsafe_allow_html=True,
    )
    score_columns = {
        column.removeprefix("score_"): result[column]
        for column in result
        if str(column).startswith("score_") and pd.notna(result[column])
    }
    if score_columns:
        score_frame = pd.DataFrame(
            [{"Sentiment": name, "Decision score": value} for name, value in score_columns.items()]
        ).sort_values("Decision score", ascending=False)
        st.dataframe(score_frame, use_container_width=True, hide_index=True)
        st.caption(
            "A decision score is the SVM margin for that class. "
            "A larger score means the review sits farther toward that class. "
            "These numbers are not probabilities."
        )


def render_bulk(pipeline):
    st.subheader("Your reviews")
    st.write(
        "Upload your own CSV. The download is that same file with one added column, "
        "`predicted_sentiment`, labeling each review Positive, Negative, or Neutral. "
        "The circle uses those labels: green for Positive, red for Negative, and blue for Neutral."
    )
    upload = st.file_uploader("Upload a CSV of reviews", type=["csv"])
    if upload is None:
        return
    if st.session_state.get("bulk_name") != upload.name:
        st.session_state.bulk_name = upload.name
        st.session_state.pop("bulk_labeled", None)
    frame = read_upload(upload)
    if frame is None:
        return
    text_column = choose_text_column(frame)
    if text_column is None:
        return
    if st.button("Label my CSV", type="primary"):
        with st.spinner("Labeling every review..."):
            predictions = predict_reviews(pipeline, frame[text_column])
        st.session_state.bulk_labeled = attach_sentiment(frame, predictions)
        st.session_state.bulk_download_name = labeled_filename(upload.name)

    labeled = st.session_state.get("bulk_labeled")
    if labeled is None:
        return
    counts = sentiment_counts(labeled["predicted_sentiment"])
    render_upload_summary(counts)
    chart_column, table_column = st.columns([0.85, 1.15], gap="large")
    with chart_column:
        st.pyplot(upload_donut(counts), clear_figure=True)
    with table_column:
        st.dataframe(labeled.head(200), use_container_width=True, hide_index=True)
        if len(labeled) > 200:
            st.caption("Showing the first 200 rows here. The download contains every row from your file.")
    st.download_button(
        "Download your labeled CSV",
        data=labeled.to_csv(index=False).encode("utf-8-sig"),
        file_name=st.session_state.get("bulk_download_name", "reviews_with_sentiment.csv"),
        mime="text/csv",
    )


def attach_sentiment(frame: pd.DataFrame, predictions: pd.DataFrame) -> pd.DataFrame:
    """Return the uploaded rows unchanged, plus one sentiment label per review."""
    labeled = frame.copy()
    labeled["predicted_sentiment"] = predictions["predicted_sentiment"].astype("string").tolist()
    notes = predictions["note"].astype(str)
    if notes.str.len().gt(0).any():
        labeled["prediction_note"] = notes.tolist()
    return labeled


def labeled_filename(upload_name: str) -> str:
    stem = Path(upload_name).stem or "reviews"
    return f"{stem}_with_sentiment.csv"


def sentiment_counts(labels: pd.Series) -> dict[str, int]:
    present = labels.dropna().astype(str)
    present = present[present.str.len() > 0]
    counts = {label: int((present == label).sum()) for label in CARD_ORDER}
    for label, count in present.value_counts().items():
        if label not in counts:
            counts[str(label)] = int(count)
    return {label: count for label, count in counts.items() if count > 0}


def render_upload_summary(counts: dict[str, int]):
    total = sum(counts.values())
    winners = majority_labels(counts)
    if len(winners) == 1:
        st.markdown(f"**Most of these reviews are {winners[0]}.** {total:,} reviews labeled.")
    else:
        joined = " and ".join(winners)
        st.markdown(f"**{joined} are tied for the most reviews.** {total:,} reviews labeled.")
    columns = st.columns(len(counts))
    for column, label in zip(columns, counts):
        color = UPLOAD_COLORS.get(label, "#1B2421")
        column.markdown(
            f"""
            <div class="stat-card">
              <p class="stat-label">{html.escape(label)}</p>
              <p class="stat-value" style="color:{color}">{counts[label]:,}</p>
            </div>
            """,
            unsafe_allow_html=True,
        )


def majority_labels(counts: dict[str, int]) -> list[str]:
    if not counts:
        return []
    highest = max(counts.values())
    return [label for label in counts if counts[label] == highest]


def upload_donut(counts: dict[str, int]):
    """Circle of the uploaded reviews. The largest sentiment fills most of the ring."""
    labels = list(counts)
    values = [counts[label] for label in labels]
    colors = [UPLOAD_COLORS.get(label, "#64748B") for label in labels]
    figure, axis = plt.subplots(figsize=(5.4, 5.4))
    axis.pie(
        values,
        colors=colors,
        startangle=90,
        counterclock=False,
        wedgeprops={"width": 0.42, "edgecolor": "white", "linewidth": 2},
    )
    total = sum(values) or 1
    winners = majority_labels(counts)
    center = winners[0] if len(winners) == 1 else "Tied"
    share = counts[winners[0]] / total * 100
    axis.text(0, 0.08, center, ha="center", va="center", fontsize=16, fontweight="bold", color="#16312B")
    axis.text(0, -0.14, f"{share:.0f}%", ha="center", va="center", fontsize=14, color="#3E514B")
    axis.legend(
        [f"{label}  {counts[label]:,}" for label in labels],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        frameon=False,
        ncol=min(len(labels), 3),
    )
    axis.set_title("Your uploaded reviews", pad=12)
    figure.patch.set_facecolor("#FFFFFF")
    figure.tight_layout()
    return figure


def render_footer(metrics):
    columns = metrics.get("columns", {})
    text_name = columns.get("text", "text")
    label_name = columns.get("label", "label")
    st.divider()
    st.caption(
        f"Pipeline: TF-IDF + LinearSVC. Text column `{text_name}`, label column `{label_name}`. "
        f"Fit time {metrics.get('fit_seconds', 'unknown')} seconds. "
        "Decision scores are margins, not probabilities."
    )


def choose_text_column(frame):
    detected = detect_text_column(frame)
    options = list(frame.columns)
    if not options:
        st.error("The uploaded CSV has no columns.")
        return None
    index = options.index(detected) if detected in options else 0
    if detected is None:
        st.info("Choose the column that contains the review text.")
    return st.selectbox("Review text column", options, index=index)


def read_upload(upload):
    for encoding in ("utf-8", "utf-8-sig", "latin-1"):
        upload.seek(0)
        try:
            frame = pd.read_csv(upload, encoding=encoding)
            break
        except UnicodeDecodeError:
            frame = None
    else:
        st.error("The file could not be read as text. Save it as a UTF-8 CSV and try again.")
        return None
    if frame is None or frame.empty:
        st.error("The uploaded CSV has no rows.")
        return None
    frame.columns = [str(column).strip() for column in frame.columns]
    return frame


def validate_review(review: str):
    cleaned = " ".join(str(review).split())
    letters = sum(character.isalpha() for character in cleaned)
    if cleaned == "":
        return "Enter a customer review before predicting."
    if letters < 3:
        return "That review is too short to classify. Add a few more words."
    return ""


def distribution_figure(metrics):
    labels = [label for label in metrics["labels"] if label in metrics["dataset"]["class_counts"]]
    values = [metrics["dataset"]["class_counts"][label] for label in labels]
    colors = [STYLES.get(label, {}).get("color", "#334155") for label in labels]
    figure, axis = plt.subplots(figsize=(6.2, 3.4))
    bars = axis.barh(labels[::-1], values[::-1], color=colors[::-1], height=0.62)
    axis.bar_label(bars, padding=4, fontsize=9)
    axis.set_title("Sentiment distribution", loc="left", fontsize=13, pad=12)
    axis.set_xlabel("Reviews")
    style_axis(figure, axis)
    return figure


def confusion_figure(metrics):
    labels = metrics["labels"]
    matrix = metrics["confusion_matrix"]
    figure, axis = plt.subplots(figsize=(6.2, 4.2))
    image = axis.imshow(matrix, cmap="Greens")
    axis.set_xticks(range(len(labels)), labels)
    axis.set_yticks(range(len(labels)), labels)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")
    axis.set_title("Test confusion matrix", loc="left", fontsize=13, pad=12)
    peak = max(max(row) for row in matrix) if matrix else 1
    for row_index, row in enumerate(matrix):
        for column_index, value in enumerate(row):
            color = "white" if value > peak * 0.55 else "#1B2421"
            axis.text(column_index, row_index, f"{value:,}", ha="center", va="center", color=color, fontsize=9)
    figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    figure.patch.set_facecolor("#FFFFFF")
    axis.set_facecolor("#FFFFFF")
    figure.tight_layout()
    return figure


def style_axis(figure, axis):
    figure.patch.set_facecolor("#FFFFFF")
    axis.set_facecolor("#FFFFFF")
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(length=0)
    figure.tight_layout()


def inject_css():
    st.markdown(
        """
        <style>
          .stApp { background: #F3F6F4; }
          .hero h1 { margin: 0; font-size: 2.4rem; letter-spacing: -0.03em; color: #16312B; }
          .hero p { color: #3E514B; font-size: 1.05rem; }
          .stat-card { background: white; border: 1px solid #E1E8E4; border-radius: 16px; padding: 0.9rem 1rem 0.7rem; min-height: 92px; }
          .stat-label { margin: 0; color: #5C6F69; font-size: 0.86rem; }
          .stat-value { margin: 0.15rem 0 0; font-size: 1.7rem; font-weight: 750; letter-spacing: -0.03em; }
          .result-banner { border-left: 8px solid; border-radius: 16px; padding: 0.9rem 1.1rem 0.7rem; margin: 0.8rem 0 0.6rem; }
          .result-kicker { margin: 0; color: #5C6F69; font-size: 0.82rem; text-transform: uppercase; letter-spacing: 0.06em; }
          .result-label { margin: 0.1rem 0; font-size: 2rem; font-weight: 750; }
          .result-score { margin: 0; color: #3E514B; }
        </style>
        """,
        unsafe_allow_html=True,
    )


if __name__ == "__main__":
    main()


## 7. Install and configure a public tunnel

This step is for Google Colab. It downloads Cloudflare's quick-tunnel program and writes a headless Streamlit config. No Cloudflare account is required. On your own computer, skip the last two cells and run `streamlit run app.py`.


In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
except ImportError:
    raise SystemExit("Tunnel setup is for Google Colab. Locally, run: streamlit run app.py")

Path(".streamlit").mkdir(exist_ok=True)
Path(".streamlit/config.toml").write_text(
    """
[server]
headless = true
address = "0.0.0.0"
port = 8501
enableCORS = false
enableXsrfProtection = false
fileWatcherType = "none"
maxUploadSize = 50

[browser]
gatherUsageStats = false
""".strip()
    + "\n",
    encoding="utf-8",
)
if not Path("cloudflared").exists():
    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            "cloudflared",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)
print("Tunnel program is ready.")


## 8. Launch the app and display its public URL

Wait until the cell prints a `trycloudflare.com` address, then open it. The first load can take a few seconds. If you run this cell again, it replaces the previous Streamlit process.


In [ ]:
import re
import subprocess
import time
from pathlib import Path

try:
    import google.colab  # noqa: F401
except ImportError:
    raise SystemExit("Launch the local app with: streamlit run app.py")

if not Path("models/sentiment_pipeline.joblib").exists():
    raise SystemExit("Save the model in the previous cells before launching the app.")

subprocess.run("pkill -f 'streamlit run app.py' || true", shell=True)
subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True)
time.sleep(1)

streamlit_log = open("streamlit.log", "w", encoding="utf-8")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)
tunnel_log = open("tunnel.log", "w", encoding="utf-8")
subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(40):
    time.sleep(1)
    tunnel_text = Path("tunnel.log").read_text(encoding="utf-8", errors="ignore")
    match = re.search(r"https://[-0-9a-zA-Z]+\.trycloudflare\.com", tunnel_text)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("Public URL:", public_url)
else:
    print("The public URL did not appear. Tunnel log:")
    print(Path("tunnel.log").read_text(encoding="utf-8", errors="ignore")[-2000:])
    print("Streamlit log:")
    print(Path("streamlit.log").read_text(encoding="utf-8", errors="ignore")[-2000:])
